
<p align="center"> In the Name of God </p>

<table style="width: 100%; text-align: center;">
  <tr>
    <td style="width: 20%;"><img src="https://i.ibb.co/yXKQmtZ/logo1.png" alt="Left Logo" width="100"></td>
    <td style="width: 60%;">
      <p>Faculty of Electrical and Computer Engineering</p>
      <p>Interactive learning - Fall 2023-2024</p>
      <p>Assignment Series 4</p>
      <p>Markov Decision Processes and Dynamic programming</p>
    </td>
    <td style="width: 20%;"><img src="https://i.ibb.co/wLjqFkw/logo2.png" alt="Right Logo" width="100"></td>
  </tr>
  
</table>
<p align="center">Fatemeh Mohammadi - 810199489</p>
<hr>



# <font color = lightGreen> PART 4
 </font>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env
from scipy.stats import sem, t

# Parameters
num_runs = 10
total_timesteps = 50000
env_name = "CartPole-v1"

# Store rewards for all runs
all_rewards = []

# Run multiple training sessions
for run in range(num_runs):
    print(f"Starting Run {run + 1}/{num_runs}")
    env = make_vec_env(env_name, n_envs=1)
    model = A2C("MlpPolicy", env, verbose=0)
    model.learn(total_timesteps=total_timesteps)

    # Record rewards from the last training episode
    episode_rewards = model.ep_info_buffer
    rewards = [info['r'] for info in episode_rewards]
    all_rewards.append(rewards)

# Pad rewards to the same length (if episodes have different lengths)
max_length = max(len(r) for r in all_rewards)
padded_rewards = np.array([np.pad(r, (0, max_length - len(r)), constant_values=np.nan) for r in all_rewards])

# Compute mean and 95% confidence interval
mean_rewards = np.nanmean(padded_rewards, axis=0)
std_err = sem(padded_rewards, axis=0, nan_policy="omit")
confidence_interval = t.ppf(0.975, df=num_runs - 1) * std_err

# Plot results
plt.figure(figsize=(10, 6))
x = np.arange(len(mean_rewards))
plt.plot(x, mean_rewards, label="Mean Reward")
plt.fill_between(x, mean_rewards - confidence_interval, mean_rewards + confidence_interval, alpha=0.2, label="95% Confidence Interval")
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("A2C Training Rewards with 95% Confidence Interval")
plt.legend()
plt.grid()
plt.show()


Starting Run 1/10


: 

In [2]:
import os
import gym
from stable_baselines3 import A2C

# Load the trained model
model = A2C.load("a2c_cartpole")

# Create a new environment with video recording
video_path = "videos/"
os.makedirs(video_path, exist_ok=True)
env = gym.wrappers.RecordVideo(
    gym.make("CartPole-v1"),
    video_path,
    episode_trigger=lambda episode_id: True,  # Record all episodes
    name_prefix="a2c_test"
)

# Run a single test episode
state = env.reset()
done = False
while not done:
    action, _ = model.predict(state, deterministic=True)
    state, _, done, _ = env.step(action)

env.close()
print(f"Video saved to {video_path}")


ModuleNotFoundError: No module named 'stable_baselines3'

In [ ]:
import torch
import gym
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt
from collections import deque

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
GAMMA = 0.99
LR = 7e-4
ENTROPY_BETA = 0.01

# Actor-Critic Model
class A2C(nn.Module):
    def __init__(self, input_dim, action_dim):
        super(A2C, self).__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU()
        )
        self.policy = nn.Sequential(
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )
        self.value = nn.Linear(128, 1)

    def forward(self, x):
        shared_out = self.shared(x)
        policy_out = self.policy(shared_out)
        value_out = self.value(shared_out)
        return policy_out, value_out

# Environment setup
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

# Initialize A2C model
model = A2C(state_dim, action_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

# Training function
def train_a2c(num_episodes):
    reward_history = []
    for episode in range(num_episodes):
        state = env.reset()
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        log_probs = []
        values = []
        rewards = []
        entropy = 0

        done = False
        while not done:
            policy, value = model(state)
            dist = Categorical(policy)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))
            values.append(value)
            entropy += dist.entropy().mean()

            next_state, reward, done, _ = env.step(action.item())
            rewards.append(reward)

            state = torch.tensor(next_state, dtype=torch.float32, device=device).unsqueeze(0)

        # Compute returns
        returns = []
        R = 0
        for reward in reversed(rewards):
            R = reward + GAMMA * R
            returns.insert(0, R)
        returns = torch.tensor(returns, dtype=torch.float32, device=device)

        # Advantage calculation
        values = torch.cat(values)
        log_probs = torch.cat(log_probs)
        advantage = returns - values.squeeze()

        # Loss calculation
        actor_loss = -(log_probs * advantage.detach()).mean()
        critic_loss = advantage.pow(2).mean()
        loss = actor_loss + 0.5 * critic_loss - ENTROPY_BETA * entropy

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        reward_history.append(sum(rewards))

        if episode % 10 == 0:
            print(f"Episode {episode}, Total Reward: {sum(rewards):.2f}")

    return reward_history

# Run training
num_episodes = 500
reward_history = train_a2c(num_episodes)

# Plot results
plt.plot(reward_history)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("A2C Training Rewards")
plt.show()

# Save video of test performance
env = gym.wrappers.RecordVideo(env, "videos", episode_trigger=lambda x: True)
state = env.reset()
state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
done = False
while not done:
    policy, _ = model(state)
    action = policy.argmax(dim=-1).item()
    next_state, _, done, _ = env.step(action)
    state = torch.tensor(next_state, dtype=torch.float32, device=device).unsqueeze(0)

env.close()


In [ ]:

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
GAMMA = 0.99
LR = 7e-4
ENTROPY_BETA = 0.01

# Actor-Critic Model
class A2C(nn.Module):
    def __init__(self, input_dim, action_dim):
        super(A2C, self).__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU()
        )
        self.policy = nn.Sequential(
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )
        self.value = nn.Linear(128, 1)

    def forward(self, x):
        if x.ndimension() == 1:  # Adjust input shape if necessary
            x = x.unsqueeze(0)
        shared_out = self.shared(x)
        policy_out = self.policy(shared_out)
        value_out = self.value(shared_out)
        return policy_out, value_out

# Environment setup
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

# Initialize A2C model
model = A2C(state_dim, action_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

# Training function
def train_a2c(num_episodes):
    reward_history = []
    for episode in range(num_episodes):
        state = env.reset()
        if isinstance(state, tuple):  # Unpack state if environment returns additional info
            state = state[0]
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

        log_probs = []
        values = []
        rewards = []
        entropy = 0

        done = False
        while not done:
            policy, value = model(state)
            dist = Categorical(policy)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))
            values.append(value)
            entropy += dist.entropy().mean()

            step_result = env.step(action.item())
            if len(step_result) == 5:  # Handle environments with additional outputs
                next_state, reward, done, truncated, info = step_result
                done = done or truncated
            else:
                next_state, reward, done, info = step_result

            if isinstance(next_state, tuple):  # Handle tuple state
                next_state = next_state[0]
            rewards.append(reward)

            state = torch.tensor(next_state, dtype=torch.float32, device=device).unsqueeze(0)

        # Compute returns
        returns = []
        R = 0
        for reward in reversed(rewards):
            R = reward + GAMMA * R
            returns.insert(0, R)
        returns = torch.tensor(returns, dtype=torch.float32, device=device)

        # Advantage calculation
        values = torch.cat(values)
        log_probs = torch.cat(log_probs)
        advantage = returns - values.squeeze()

        # Loss calculation
        actor_loss = -(log_probs * advantage.detach()).mean()
        critic_loss = advantage.pow(2).mean()
        loss = actor_loss + 0.5 * critic_loss - ENTROPY_BETA * entropy

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        reward_history.append(sum(rewards))

        if episode % 10 == 0:
            print(f"Episode {episode}, Total Reward: {sum(rewards):.2f}")

    return reward_history

# Run training
num_episodes = 500
reward_history = train_a2c(num_episodes)

# Plot results
plt.plot(reward_history)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("A2C Training Rewards")
plt.show()


In [ ]:

# Hyperparameters
GAMMA = 0.99
LR = 7e-4
ENTROPY_BETA = 0.01

# Actor-Critic Model
class A2C(nn.Module):
    def __init__(self, input_dim, action_dim):
        super(A2C, self).__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU()
        )
        self.policy = nn.Sequential(
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )
        self.value = nn.Linear(128, 1)

    def forward(self, x):
        if x.ndimension() == 1:  # Adjust input shape if necessary
            x = x.unsqueeze(0)
        shared_out = self.shared(x)
        policy_out = self.policy(shared_out)
        value_out = self.value(shared_out)
        return policy_out, value_out

# Environment setup
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

# Initialize A2C model
model = A2C(state_dim, action_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

# Training function
def train_a2c(num_episodes):
    reward_history = []
    for episode in range(num_episodes):
        state = env.reset()
        if isinstance(state, tuple):  # Unpack state if environment returns additional info
            state = state[0]
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

        log_probs = []
        values = []
        rewards = []
        entropy = 0

        done = False
        while not done:
            policy, value = model(state)
            dist = Categorical(policy)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))
            values.append(value)
            entropy += dist.entropy().mean()

            step_result = env.step(action.item())
            if len(step_result) == 5:  # Handle environments with additional outputs
                next_state, reward, done, truncated, info = step_result
                done = done or truncated
            else:
                next_state, reward, done, info = step_result

            if isinstance(next_state, tuple):  # Handle tuple state
                next_state = next_state[0]
            rewards.append(reward)

            state = torch.tensor(next_state, dtype=torch.float32, device=device).unsqueeze(0)

        # Compute returns
        returns = []
        R = 0
        for reward in reversed(rewards):
            R = reward + GAMMA * R
            returns.insert(0, R)
        returns = torch.tensor(returns, dtype=torch.float32, device=device)

        # Advantage calculation
        values = torch.cat(values)
        log_probs = torch.cat(log_probs)
        advantage = returns - values.squeeze()

        # Loss calculation
        actor_loss = -(log_probs * advantage.detach()).mean()
        critic_loss = advantage.pow(2).mean()
        loss = actor_loss + 0.5 * critic_loss - ENTROPY_BETA * entropy

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        reward_history.append(sum(rewards))

        if episode % 10 == 0:
            print(f"Episode {episode}, Total Reward: {sum(rewards):.2f}")

    return reward_history

# Run training
num_episodes = 500
reward_history = train_a2c(num_episodes)

# Plot results
plt.plot(reward_history)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("A2C Training Rewards")
plt.show()


In [ ]:

# Actor-Critic Model
class A2C(nn.Module):
    def __init__(self, input_dim, action_dim):
        super(A2C, self).__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU()
        )
        self.policy = nn.Sequential(
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )
        self.value = nn.Linear(128, 1)

    def forward(self, x):
        if x.ndimension() == 1:  # Adjust input shape if necessary
            x = x.unsqueeze(0)
        shared_out = self.shared(x)
        policy_out = self.policy(shared_out)
        value_out = self.value(shared_out)
        return policy_out, value_out

# Environment setup
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

# Initialize A2C model
model = A2C(state_dim, action_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

# Training function
def train_a2c(num_episodes):
    reward_history = []
    for episode in range(num_episodes):
        state = env.reset()
        if isinstance(state, tuple):  # Unpack state if environment returns additional info
            state = state[0]
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

        log_probs = []
        values = []
        rewards = []
        entropy = 0

        done = False
        while not done:
            policy, value = model(state)
            dist = Categorical(policy)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))
            values.append(value)
            entropy += dist.entropy().mean()

            step_result = env.step(action.item())
            if len(step_result) == 5:  # Handle environments with additional outputs
                next_state, reward, done, truncated, info = step_result
                done = done or truncated
            else:
                next_state, reward, done, info = step_result

            if isinstance(next_state, tuple):  # Handle tuple state
                next_state = next_state[0]
            rewards.append(reward)

            state = torch.tensor(next_state, dtype=torch.float32, device=device).unsqueeze(0)

        # Compute returns
        returns = []
        R = 0
        for reward in reversed(rewards):
            R = reward + GAMMA * R
            returns.insert(0, R)
        returns = torch.tensor(returns, dtype=torch.float32, device=device)

        # Advantage calculation
        values = torch.cat(values)
        log_probs = torch.cat(log_probs)
        advantage = returns - values.squeeze()

        # Loss calculation
        actor_loss = -(log_probs * advantage.detach()).mean()
        critic_loss = advantage.pow(2).mean()
        loss = actor_loss + 0.5 * critic_loss - ENTROPY_BETA * entropy

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        reward_history.append(sum(rewards))

        if episode % 10 == 0:
            print(f"Episode {episode}, Total Reward: {sum(rewards):.2f}")

    return reward_history

# Run training
num_episodes = 500
reward_history = train_a2c(num_episodes)

# Plot results
plt.plot(reward_history)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("A2C Training Rewards")
plt.show()


In [ ]:

env = gym.wrappers.RecordVideo(env, "videos", episode_trigger=lambda x: True)
state = env.reset()
if isinstance(state, tuple):  # Unpack state if environment returns additional info
    state = state[0]
state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
done = False
while not done:
    policy, _ = model(state)
    action = policy.argmax(dim=-1).item()
    step_result = env.step(action)
    if len(step_result) == 5:  # Handle environments with additional outputs
        next_state, _, done, truncated, _ = step_result
        done = done or truncated
    else:
        next_state, _, done, _ = step_result
    if isinstance(next_state, tuple):  # Handle tuple state
        next_state = next_state[0]
    state = torch.tensor(next_state, dtype=torch.float32, device=device).unsqueeze(0)

env.close()
